# Natural Language Processing

Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to enable computers to understand, interpret, and generate human language in a way that is valuable.

Here, we will cover several key topics in NLP, including:

- n-gram language modeling and smoothing
- bag-of-words classification
- part-of-speech tagging with Viterbi decoding
- probabilistic context-free grammar parsing
- augmented grammars with agreement features
- semantic interpretation
- quantifier-scope ambiguity
- information extraction and simple question answering

## Setup

Run this cell first. The notebook uses only the Python standard library.

In [1]:
import math
import random
import re
from collections import Counter, defaultdict

random.seed(11)


def show_rows(rows, headers=None):
    # This function prints a list of rows in a tabular format. Each row is a list of values, 
    # and headers is an optional list of column names.
    if headers:
        print(" | ".join(headers))
        print("-" * (3 + sum(len(h) for h in headers)))
    for row in rows:
        print(" | ".join(str(x) for x in row))


corpus = [
    "the wumpus is near .",
    "the pit is near .",
    "the gold is glittering .",
    "the agent grabs the gold .",
    "the agent smells the wumpus .",
    "the breeze means the pit is near .",
    "the stench means the wumpus is near .",
]

## Tokenization and N-gram Language Models

Implement a lowercase tokenizer, then train a bigram or trigram model from the toy corpus.

Requirements:

- `tokenize(text)` should return words and sentence punctuation such as `"."`
- `train_ngram(sentences, n)` should count contexts and n-grams
- `next_word_probability(...)` should support add-k smoothing
- `sentence_log_probability(...)` should return `-inf` when an unsmoothed probability is zero

In [2]:
def tokenize(text):
    # Return lowercase tokens. Keep ".", "!", and "?" as separate tokens.
    # Clue: Use regex to find sequences of letters (a-z) and apostrophes, or punctuation marks.
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[.!?]", text.lower())


def train_ngram(sentences, n):
    # Return a dictionary with keys: n, context_counts, ngram_counts.
    # Hint: pad each sentence with n-1 start symbols.
    # Count the occurrences of each context and n-gram in the sentences.
    # For example, for n=2, the context is the previous word, and for n=3, the context is the previous two words.
    context_counts = Counter()
    ngram_counts = Counter()

    for sent in sentences:
        # With n=3, the sentence "the wumpus is near ." becomes "<s> <s> the wumpus is near ."
        padded = ["<s>"] * (n - 1) + [w for w in sent if w != "<s>"]
        for i in range(n - 1, len(padded)):
            # The context is the n-1 tokens immediately before the word being processed.
            context = tuple(padded[i - n + 1:i])
            word = padded[i]
            context_counts[context] += 1
            ngram_counts[context + (word,)] += 1

    return {"n": n, "context_counts": context_counts, "ngram_counts": ngram_counts}


def next_word_probability(model, context, word, vocabulary, k=0.0):
    # Estimate P(word | context) with add-k smoothing.
    # Add-k smoothing adds k to the numerator and k * |V| to the denominator to avoid zero probabilities.
    # Cutting the context to the right length for the n-gram model.
    n = model["n"]
    # Only keep the last n-1 tokens of the context, as required by the n-gram model.
    context = tuple(context[-(n - 1):]) if n > 1 else tuple()

    # Add-k smoothing adds k to the n-gram count and k * |V| to the context count.
    numerator = model["ngram_counts"][context + (word,)] + k
    denominator = model["context_counts"][context] + k * len(vocabulary)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_log_probability(model, sentence, vocabulary, k=0.0):
    # Sum log probabilities for each generated token, including </s>.
    # Use add-k smoothing to avoid zero probabilities. If k=0 and a zero probability is encountered, return float("-inf").
    # If no smoothing is used (k=0), and a zero probability is encountered, return float("-inf").
    n = model["n"]
    padded = ["<s>"] * (n - 1) + tokenize(sentence) + ["</s>"]
    logp = 0.0
    details = []
    for i in range(n - 1, len(padded)):
        context = tuple(padded[i - n + 1:i])
        word = padded[i]
        p = next_word_probability(model, context, word, vocabulary, k=k)
        details.append((context, word, p))
        if p == 0:
            return float("-inf"), details

        # Use log probabilities to avoid underflow issues when multiplying many small probabilities together.
        logp += math.log(p)
    return logp, details

In [3]:
# Add sentence start/end markers so the model can learn sentence boundaries.
tokenized = [["<s>"] + tokenize(sentence) + ["</s>"] for sentence in corpus]

# A set comprehension gathers all distinct tokens; sorting keeps the vocabulary order stable.
vocab = sorted({word for sent in tokenized for word in sent})
bigram = train_ngram(tokenized, n=2)

# The asserts below are automatic tests: if any fail, Python stops at the corresponding assert line.
assert tokenize("The Wumpus is near.") == ["the", "wumpus", "is", "near", "."]
assert bigram["context_counts"]["the",] == 11

p_is_after_wumpus = next_word_probability(bigram, ("wumpus",), "is", vocab, k=0.0)
assert abs(p_is_after_wumpus - (2 / 3)) < 1e-9

# This sentence contains a bigram never seen before, so the unsmoothed version should assign a log probability of -inf.
raw_logp, raw_details = sentence_log_probability(bigram, "the agent is glittering .", vocab, k=0.0)
smooth_logp, smooth_details = sentence_log_probability(bigram, "the agent is glittering .", vocab, k=0.5)
assert raw_logp == float("-inf")
assert smooth_logp > float("-inf")

print("Tests passed.")

Tests passed.


## Bag-of-Words and Naive Bayes

A bag-of-words model ignores word order. Implement a small multinomial Naive Bayes classifier and observe one limitation: `"not good"` still contains the positive clue `"good"`.

In [4]:
train_docs = [
    ("good fun bright", "positive"),
    ("good helpful clear", "positive"),
    ("excellent fun useful", "positive"),
    ("bad boring dull", "negative"),
    ("bad confusing slow", "negative"),
    ("dull boring useless", "negative"),
]


def bow_vector(text):
# : Return Counter({word: count}) for alphabetic tokens only.
    # Counter is useful for bag-of-words: it counts occurrences without storing word order.
    # Hint: filter out tokens that do not begin with a letter using re.match(...).
    return Counter(w for w in tokenize(text) if re.match(r"[a-z]", w))


def train_multinomial_nb(examples):
# : Return enough statistics for prediction:
    # labels, label_counts, word_counts_by_label, total_words_by_label, vocabulary.
    # Hint: count the prior P(label) and the likelihood P(word | label) for each label.
    # word_counts_by_label should be a dictionary mapping label -> Counter(words).
    labels = sorted({label for _, label in examples})
    label_counts = Counter(label for _, label in examples)
    word_counts = {label: Counter() for label in labels}
    total_words = Counter()
    vocab = set()

    for text, label in examples:
        words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]

        # Store the number of times each word appears under each label to estimate P(word | label).
        word_counts[label].update(words)
        total_words[label] += len(words)
        vocab.update(words)

    return labels, label_counts, word_counts, total_words, sorted(vocab)


def nb_predict(model, text, alpha=1.0):
# : Return (best_label, score_dictionary).
    # Hint: compute log prior + total log likelihood for each label, then choose the label with the highest score.
    # alpha is the Laplace smoothing factor so unseen words do not make the probability zero.
    labels, label_counts, word_counts, total_words, vocab = model
    words = [w for w in tokenize(text) if re.match(r"[a-z]", w)]
    scores = {}
    for label in labels:
        # Start with the log prior P(label).
        score = math.log(label_counts[label] / sum(label_counts.values()))
        for word in words:
            # Add the log likelihood with Laplace smoothing so rare or unseen words do not make the probability zero.
            score += math.log((word_counts[label][word] + alpha) / (total_words[label] + alpha * len(vocab)))
        scores[label] = score
    return max(scores, key=scores.get), scores

In [5]:
assert bow_vector("Dog bites dog!") == Counter({"dog": 2, "bites": 1})

nb_model = train_multinomial_nb(train_docs)

# These two tests check that the classifier correctly labels sentences with clear keywords.
label, scores = nb_predict(nb_model, "good useful fun")
assert label == "positive"

label, scores = nb_predict(nb_model, "bad dull slow")
assert label == "negative"

# This example shows the limitation of bag-of-words: "not good" still contains the word "good".
print("Prediction for 'not good':", nb_predict(nb_model, "not good")[0])
print("Tests passed.")

Prediction for 'not good': positive
Tests passed.


## POS Tagging with Viterbi Decoding

Implement Viterbi decoding for a tiny hidden Markov model. The model should choose a likely tag sequence for `"time flies like an arrow"`.

In [6]:
states = ["N", "V", "DET", "P"]

# start_prob: the probability of the first tag in the sentence.
start_prob = {"N": 0.35, "V": 0.10, "DET": 0.45, "P": 0.10}

# transition_prob[prev][state]: the probability of transitioning from the previous tag to the current tag.
transition_prob = {
    "N": {"N": 0.05, "V": 0.55, "DET": 0.05, "P": 0.20, "END": 0.15},
    "V": {"N": 0.20, "V": 0.05, "DET": 0.35, "P": 0.30, "END": 0.10},
    "DET": {"N": 0.90, "V": 0.02, "DET": 0.02, "P": 0.03, "END": 0.03},
    "P": {"N": 0.35, "V": 0.03, "DET": 0.57, "P": 0.02, "END": 0.03},
}

# emission_prob[state][word]: the probability that the tag emits the observed word.
emission_prob = {
    "N": {"time": 0.30, "flies": 0.20, "arrow": 0.30, "fruit": 0.20},
    "V": {"time": 0.15, "flies": 0.30, "like": 0.30, "saw": 0.25},
    "DET": {"an": 0.50, "the": 0.50},
    "P": {"like": 0.80, "with": 0.20},
}


def safe_log(x):
    # Return -inf for zero probability so the log-sum still represents impossible paths.
    return math.log(x) if x > 0 else float("-inf")


def viterbi(words):
# : Return (tagged_words, best_log_score).
    # tagged_words should look like [("time", "N"), ("flies", "V"), ...].
    # Hint: table stores the best score for each state at each position.
    # back stores the previous state to trace back to the best tag sequence after completion.
    table = []
    back = []

    first = {}
    first_back = {}
    for state in states:
        emission = emission_prob[state].get(words[0], 0.001)

        # The initial score is the start probability for that tag times the emission probability of the first word.
        first[state] = safe_log(start_prob[state]) + safe_log(emission)
        first_back[state] = None
    table.append(first)
    back.append(first_back)

    for i in range(1, len(words)):
        row = {}
        row_back = {}
        for state in states:
            emission = emission_prob[state].get(words[i], 0.001)
            candidates = []
            for prev in states:
                # Viterbi keeps only the best path to each state instead of enumerating every tag sequence.
                candidates.append((table[i - 1][prev] + safe_log(transition_prob[prev][state]) + safe_log(emission), prev))
            row[state], row_back[state] = max(candidates)
        table.append(row)
        back.append(row_back)

    final_candidates = [(table[-1][state] + safe_log(transition_prob[state]["END"]), state) for state in states]
    best_score, best_state = max(final_candidates)

    # Trace backward through the backpointer to recover the best tag sequence.
    tags = [best_state]
    for i in range(len(words) - 1, 0, -1):
        tags.append(back[i][tags[-1]])
    tags.reverse()
    return list(zip(words, tags)), best_score

In [7]:
tagged, score = viterbi(tokenize("time flies like an arrow"))
print(tagged)

# This test checks the best tag sequence that Viterbi must find from the given probabilistic model.
assert tagged == [("time", "N"), ("flies", "V"), ("like", "P"), ("an", "DET"), ("arrow", "N")]
assert isinstance(score, float)
print("Tests passed.")

[('time', 'N'), ('flies', 'V'), ('like', 'P'), ('an', 'DET'), ('arrow', 'N')]
Tests passed.


## PCFG and Bottom-Up Chart Parsing

Implement a CYK-like chart parser for this small probabilistic context-free grammar. The sentence
`"i saw the wumpus with the telescope"` should have two parses because the prepositional phrase can attach in two different places.

In [8]:
pcfg_rules = [
    ("S",  ("NP", "VP"), 1.00),
    ("NP", ("Det", "N"), 0.45),
    ("NP", ("NP", "PP"), 0.25),
    ("NP", ("i",), 0.30),
    ("VP", ("V", "NP"), 0.60),
    ("VP", ("VP", "PP"), 0.40),
    ("PP", ("P", "NP"), 1.00),
    ("Det", ("the",), 1.00),
    ("N", ("wumpus",), 0.35),
    ("N", ("telescope",), 0.25),
    ("N", ("pit",), 0.20),
    ("N", ("gold",), 0.20),
    ("V", ("saw",), 0.70),
    ("V", ("smelled",), 0.30),
    ("P", ("with",), 0.60),
    ("P", ("near",), 0.40),
]


def split_rules(rules):
# : Split grammar rules into lexical and binary rules.
    # Return (lexical_rules, binary_rules).
    # Hint: lexical rules have the form A -> word; binary rules have the form A -> B C.
    # Convert probabilities to math.log(prob) so scores can be added when combining parse trees.
    nonterminals = {lhs for lhs, _, _ in rules}
    lexical = []
    binary = []
    for lhs, rhs, prob in rules:
        if len(rhs) == 1 and rhs[0] not in nonterminals:
            lexical.append((lhs, rhs[0], math.log(prob)))
        elif len(rhs) == 2:
            binary.append((lhs, rhs[0], rhs[1], math.log(prob)))
        else:
            raise ValueError(f"Unsupported rule: {lhs} -> {rhs}")
    return lexical, binary


def tree_to_string(tree, indent=0):
    label, children = tree
    pad = "  " * indent
    if isinstance(children, str):
        return f"{pad}({label} {children})"

    # Recursively print the parse tree: each child node is indented one level deeper.
    lines = [f"{pad}({label}"]
    for child in children:
        lines.append(tree_to_string(child, indent + 1))
    lines[-1] += ")"
    return "\n".join(lines)


def chart_parse(words, lexical_rules, binary_rules, keep_per_symbol=5):
# : Fill a chart over spans and keep the highest-scoring parses.
    # Hint: chart[i][j] should store the best parses for the span words[i:j].
    # For each span, try every split point k and every binary rule A -> B C to combine the left and right subtrees.
    # keep_per_symbol limits how many trees are kept for each nonterminal so the chart does not grow too large.
    n = len(words)

    # chart[i][j] stores the best parse trees for the span words[i:j].
    chart = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n)]

    for i, word in enumerate(words):
        for lhs, terminal, logprob in lexical_rules:
            if terminal == word:
                chart[i][i + 1][lhs].append((logprob, (lhs, word)))

    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                # Try every split point k to combine the left and right constituents.
                for lhs, left, right, rule_logprob in binary_rules:
                    for left_logprob, left_tree in chart[i][k].get(left, []):
                        for right_logprob, right_tree in chart[k][j].get(right, []):
                            total = rule_logprob + left_logprob + right_logprob
                            chart[i][j][lhs].append((total, (lhs, [left_tree, right_tree])))

            for symbol in list(chart[i][j].keys()):
                # Keep only the best few trees for each nonterminal so the table does not grow too large.
                chart[i][j][symbol].sort(key=lambda item: item[0], reverse=True)
                chart[i][j][symbol] = chart[i][j][symbol][:keep_per_symbol]

    return chart

In [9]:
lexical_rules, binary_rules = split_rules(pcfg_rules)
words = tokenize("i saw the wumpus with the telescope")
chart = chart_parse(words, lexical_rules, binary_rules)
parses = chart[0][len(words)]["S"]

# This sentence can attach the prepositional phrase "with the telescope" in two ways, so there should be 2 parses.
assert len(parses) == 2

# The first parse must have a higher log probability than the second parse.
assert parses[0][0] > parses[1][0]

for rank, (logprob, tree) in enumerate(parses, 1):
    print("\nParse", rank, "log probability:", round(logprob, 3))
    print(tree_to_string(tree))

print("Tests passed.")


Parse 1 log probability: -7.532
(S
  (NP i)
  (VP
    (VP
      (V saw)
      (NP
        (Det the)
        (N wumpus)))
    (PP
      (P with)
      (NP
        (Det the)
        (N telescope)))))

Parse 2 log probability: -8.002
(S
  (NP i)
  (VP
    (V saw)
    (NP
      (NP
        (Det the)
        (N wumpus))
      (PP
        (P with)
        (NP
          (Det the)
          (N telescope))))))
Tests passed.


## Augmented Grammar with Agreement Features

A plain CFG may accept sentences such as `"she eat bananas"`. Add agreement features so the checker accepts only compatible subject-verb pairs.

In [10]:
pronoun_features = {
    "i": {"person_number": "1sg"},
    "you": {"person_number": "2"},
    "we": {"person_number": "pl"},
    "they": {"person_number": "pl"},
    "she": {"person_number": "3sg"},
    "he": {"person_number": "3sg"},
}

# Each verb stores the set of subject features it may take.
verb_features = {
    "eat": {"allowed_subjects": {"1sg", "2", "pl"}},
    "eats": {"allowed_subjects": {"3sg"}},
    "see": {"allowed_subjects": {"1sg", "2", "pl"}},
    "sees": {"allowed_subjects": {"3sg"}},
}

nouns = {"bananas", "gold", "wumpus", "pit"}


def agreement_parse(sentence):
# : Return (accepted_bool, explanation_string).
    # Hint: split the sentence into subject, verb, and object; check whether each word is in the lexicon.
    # Then compare the subject's person_number with the verb's allowed_subjects.
    words = tokenize(sentence)
    if len(words) != 3:
        return False, "Expected: pronoun verb object"

    subj, verb, obj = words
    if subj not in pronoun_features:
        return False, "Unknown subject"
    if verb not in verb_features:
        return False, "Unknown verb"
    if obj not in nouns:
        return False, "Unknown object"

    # Match the subject's person-number features with the set of subject features the verb allows.
    subject_pn = pronoun_features[subj]["person_number"]
    allowed = verb_features[verb]["allowed_subjects"]
    if subject_pn not in allowed:
        return False, f"Agreement failure: subject is {subject_pn}, but verb '{verb}' allows {sorted(allowed)}"

    return True, f"Valid: NP({subject_pn}) + VP({verb})"

In [11]:
# These assertions check both grammatically correct and incorrect agreement cases.
assert agreement_parse("i eat bananas")[0] is True
assert agreement_parse("she eats bananas")[0] is True
assert agreement_parse("she eat bananas")[0] is False
assert agreement_parse("they sees gold")[0] is False

for sent in ["i eat bananas", "she eats bananas", "she eat bananas", "they sees gold"]:
    print(sent, "->", agreement_parse(sent))

print("Exercise 5 tests passed.")

i eat bananas -> (True, 'Valid: NP(1sg) + VP(eat)')
she eats bananas -> (True, 'Valid: NP(3sg) + VP(eats)')
she eat bananas -> (False, "Agreement failure: subject is 3sg, but verb 'eat' allows ['1sg', '2', 'pl']")
they sees gold -> (False, "Agreement failure: subject is pl, but verb 'sees' allows ['3sg']")
Exercise 5 tests passed.


## Semantic Interpretation

Implement a tiny compositional interpreter for subject-verb-object sentences. The output should be a logical form string such as `Loves(Ali, Bo)`.

In [12]:
names = {"ali": "Ali", "bo": "Bo", "cy": "Cy"}


def love_relation(obj):
    # The function returns a lambda representing the idea: the verb takes an object and then waits for the subject.
    return lambda subj: f"Loves({subj}, {obj})"


def see_relation(obj):
    return lambda subj: f"Sees({subj}, {obj})"


verb_meanings = {
    "loves": love_relation,
    "sees": see_relation,
}


def interpret_svo(sentence):
# Convert "ali loves bo" into "Loves(Ali, Bo)".
    # Hint: tokenize the sentence into 3 parts, look up names in names, and then apply verb_meanings[verb](object)(subject).
    words = tokenize(sentence)
    if len(words) != 3:
        raise ValueError("Expected exactly: Name Verb Name")
    subj_word, verb_word, obj_word = words
    subj = names[subj_word]
    obj = names[obj_word]

    # Apply the verb meaning using compositional semantics: verb(object)(subject).
    return verb_meanings[verb_word](obj)(subj)

In [13]:
assert interpret_svo("ali loves bo") == "Loves(Ali, Bo)"
assert interpret_svo("bo loves ali") == "Loves(Bo, Ali)"
assert interpret_svo("cy sees bo") == "Sees(Cy, Bo)"

facts = {"Loves(Ali, Bo)", "Sees(Cy, Bo)"}

# After the sentence is converted into a logical form, checking correctness is a set-membership test against the facts.
print("Is 'ali loves bo' true?", interpret_svo("ali loves bo") in facts)
print("Tests passed.")

Is 'ali loves bo' true? True
Tests passed.


## Quantifier-Scope Ambiguity

The sentence `"Every robot saw a pit"` has at least two readings:

- narrow-scope existential: every robot saw at least one pit, possibly different pits
- wide-scope existential: there is one pit that every robot saw

Implement both truth conditions.

In [14]:
robots = {"r1", "r2"}
pits = {"p1", "p2"}

world_a = {("r1", "p1"), ("r2", "p2")}
world_b = {("r1", "p1"), ("r2", "p1")}


def every_robot_saw_a_pit_narrow(world):
# Return True if each robot saw at least one pit.
    # Hint: use all(...) for "every robot" and any(...) for "a pit".
    # In narrow scope, each robot may see a different pit.
    return all(any((robot, pit) in world for pit in pits) for robot in robots)



def every_robot_saw_a_pit_wide(world):
# Return True if there exists one pit that every robot saw.
    # Hint: use any(...) for "there exists one pit" wrapped around all(...) for "every robot".
    # In wide scope, there must be the same pit seen by all robots.
    return any(all((robot, pit) in world for robot in robots) for pit in pits)

In [15]:
# world_a is correct under narrow scope but wrong under wide scope because the two robots see different pits.
assert every_robot_saw_a_pit_narrow(world_a) is True
assert every_robot_saw_a_pit_wide(world_a) is False

# world_b is correct under both readings because both robots see p1.
assert every_robot_saw_a_pit_narrow(world_b) is True
assert every_robot_saw_a_pit_wide(world_b) is True

print("Tests passed.")

Tests passed.


## Information Extraction and Question Answering

Extract structured facts from short documents. Then answer questions from those facts.

In [16]:
documents = [
    "Alice grabbed the gold in room1.",
    "Bob smelled the wumpus in room2.",
    "Alice saw a pit in room3.",
    "The breeze means a pit is near room3.",
]

patterns = [
    # Named groups (?P<name>...) help extract the agent/action/object/location directly from the sentence.
    ("event", re.compile(r"(?P<agent>[A-Z][a-z]+) (?P<action>grabbed|smelled|saw) (?:the|a) (?P<object>[a-z]+) in (?P<location>room[0-9]+)", re.I)),
    ("clue", re.compile(r"the (?P<clue>breeze|stench) means (?:the|a) (?P<object>[a-z]+) is near (?P<location>room[0-9]+)", re.I)),
]


def extract_facts(docs):
# : Return a list of dictionaries for event facts and clue facts.
    # Event example:
    # {"kind": "event", "agent": "alice", "action": "grabbed", "object": "gold", "location": "room1"}
    # Clue example:
    # {"kind": "clue", "clue": "breeze", "object": "pit", "location": "room3"}
    # Hint: use re.compile(...) with named groups (?P<name>...) to extract the fields to save.
    # groupdict() can convert named groups into a dictionary to create facts more quickly.
    facts = []
    for doc in docs:
        for kind, pattern in patterns:
            match = pattern.search(doc)
            if match:
                # groupdict() converts named groups into a dictionary; ** is used to merge them into the new fact.
                item = {"kind": kind, **{k: v.lower() for k, v in match.groupdict().items()}}
                facts.append(item)
                break
    return facts

def answer_question(question, facts):
# Support:
    # "What did Alice grab?"
    # "Where did Bob smell the wumpus?"
    # "What is near room3?"
    # Hint: normalize the question to lowercase and use re.match(...) to recognize the question pattern.
    # Then filter the list of facts by list comprehension to get the matching object/location.
    q = question.lower()

    m = re.match(r"what did ([a-z]+) (grab|smell|see)\?", q)
    if m:
        agent, action = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]

        # The list comprehension filters the facts matching the agent and action, then selects the object as the answer.
        answers = [f["object"] for f in facts if f.get("agent") == agent and f.get("action") == wanted]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"where did ([a-z]+) (grab|smell|see) (?:the|a) ([a-z]+)\?", q)
    if m:
        agent, action, obj = m.groups()
        action_map = {"grab": "grabbed", "smell": "smelled", "see": "saw"}
        wanted = action_map[action]
        answers = [
            f["location"]
            for f in facts
            if f.get("agent") == agent and f.get("action") == wanted and f.get("object") == obj
        ]
        return ", ".join(answers) if answers else "I do not know."

    m = re.match(r"what is near (room[0-9]+)\?", q)
    if m:
        location = m.group(1)
        answers = [f["object"] for f in facts if f["kind"] == "clue" and f["location"] == location]
        return ", ".join(answers) if answers else "I do not know."

    return "I do not understand the question pattern."


In [17]:
facts = extract_facts(documents)
assert len(facts) == 4

# These tests check both questions that have an answer and questions without a matching fact.
assert answer_question("What did Alice grab?", facts) == "gold"
assert answer_question("Where did Bob smell the wumpus?", facts) == "room2"
assert answer_question("What is near room3?", facts) == "pit"
assert answer_question("What did Bob grab?", facts) == "I do not know."

for fact in facts:
    print(fact)

print("Tests passed.")

{'kind': 'event', 'agent': 'alice', 'action': 'grabbed', 'object': 'gold', 'location': 'room1'}
{'kind': 'event', 'agent': 'bob', 'action': 'smelled', 'object': 'wumpus', 'location': 'room2'}
{'kind': 'event', 'agent': 'alice', 'action': 'saw', 'object': 'pit', 'location': 'room3'}
{'kind': 'clue', 'clue': 'breeze', 'object': 'pit', 'location': 'room3'}
Tests passed.


## Reflection Questions

1. **Why does smoothing matter for n-gram language models?**
    Smoothing is crucial because it addresses the problem of zero probabilities for unseen n-grams in the training data. Without smoothing, any n-gram not present in the corpus would have a probability of zero, leading to an overall sentence probability of zero (or log probability of negative infinity), which is unrealistic and makes the model unable to handle new sequences.

2. **What kind of information is lost in a bag-of-words representation?**
    A bag-of-words representation loses all information about word order and grammatical structure within a sentence. This means that important aspects like negation ("not good" vs. "good") or semantic relationships conveyed by word sequence are discarded, which can lead to misinterpretations.

3. **Why can a sentence have more than one parse tree?**
    A sentence can have more than one parse tree due to syntactic ambiguity. This occurs when a sentence's structure allows for multiple valid interpretations based on the grammar rules, such as different ways to attach a prepositional phrase or coordinate conjunctions.

4. **What do augmented grammars add beyond plain CFG categories?**
    Augmented grammars add features (like person, number, gender, tense) and unification constraints to plain Context-Free Grammar (CFG) categories. These features allow the grammar to capture linguistic phenomena like subject-verb agreement or case marking, enabling more precise syntactic and semantic analysis by enforcing dependencies between words beyond simple category matching.

5. **Why do real NLP systems usually combine statistical, syntactic, semantic, and pragmatic evidence?**
    Real NLP systems combine these different types of evidence to achieve a more robust and accurate understanding of human language, which is inherently complex and ambiguous. Statistical methods handle variability and unknown patterns, syntactic analysis provides structural understanding, semantics deals with meaning, and pragmatics considers context and intent, all of which are necessary for comprehensive language processing.